# Сверка: Excel апрель vs `MPOS_RENT.N_AMT`

Узкий скрипт сверки `commission_monthly`:
- **Lake:** `ods_alpha.scd1_mrc_pos_rent.n_amt` (зерно мерчанта `c_nmrc`, период `d_rent`);
- **Excel:** `04_Апрель_2026.xlsx` (зерно `ИНН + ID договора`);
- **Ключ сравнения:** `inn_key + agr_id_key` после маппинга `c_nmrc -> agr_terms -> agreements -> companies`.

Итог: coverage, totals, exact/near match rate, TOP-200 расхождений, Excel-отчёт.

In [ ]:
import re
import time
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )

## 0) Конфиг

In [ ]:
# === Таблицы (MPOS_RENT = scd1_mrc_pos_rent) ===
mrc_table = 'ods_alpha.scd1_mrc_pos_rent'
agr_terms_table = 'ods_alpha.scd1_agr_terms'
agreements_table = 'ods_alpha.scd1_agreements'
companies_table = 'ods_alpha.scd1_companies'

# === Период ===
month_label = '2026-04'
month_start = '2026-04-01'
month_end = '2026-04-30'

# === Excel ===
excel_april_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_header_april = 0  # если заголовок на второй строке — поставьте 1

# === Пороги match ===
exact_abs_tol = 0.01
near_abs_tol = 1.0
near_pct_tol = 1.0  # %

# === Подключение ===
impala_db = 'sandbox_ai'
impala_mem_limit = '8g'
impala_user_name = 'Shestopalov-VYur'

# === Выгрузка ===
output_dir = Path('/home/jovyan/documents/Equaring/Data')
output_report_path = output_dir / 'mpos_rent_april_excel_compare.xlsx'

print('month:', month_label, month_start, '..', month_end)
print('mrc_table:', mrc_table)
print('excel:', excel_april_path, f'(header={excel_header_april})')
print('output:', output_report_path)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user_name}
)
imp._init_connection()


def run_sql(sql_text, step_name='query', mem_limit=impala_mem_limit):
    start_ts = time.perf_counter()
    print(f'[{step_name}] start')
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        df = imp.fetch(sql_text)
    elapsed = round(time.perf_counter() - start_ts, 2)
    rows = len(df) if isinstance(df, pd.DataFrame) else 0
    print(f'[{step_name}] done in {elapsed}s, rows={rows:,}')
    return df


print('Impala connection initialized')

## 1) Доступ к таблице

In [ ]:
sql_access = f"""
select 1 as probe_ok
from {mrc_table}
limit 1
"""

access_ok = False
access_error = None

try:
    access_df = run_sql(sql_access, step_name='access_check')
    access_ok = True
    print('ACCESS_OK')
    display(access_df)
except Exception as exc:
    access_error = f'{type(exc).__name__}: {exc}'
    print('ACCESS_ERROR:', access_error)

display(pd.DataFrame([{'table': mrc_table, 'access_ok': access_ok, 'error': access_error}]))

## 2) Lake: дедуп `n_amt` + маппинг `c_nmrc -> inn+agr_id` + coverage

In [ ]:
april_mapping_cte = f"""
with rent_base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        cast(ods_commit_ts as timestamp) as ods_commit_ts,
        cast(ods_insert_ts as timestamp) as ods_insert_ts,
        cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
    where c_nmrc is not null
      and cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
), rent_ranked as (
    select
        *,
        row_number() over (
            partition by c_nmrc, d_rent_dt
            order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
        ) as rn
    from rent_base
    where ods_deleted_flg not in ('1', 'Y', 'y')
), rent_dedup as (
    select c_nmrc, d_rent_dt, n_amt_num
    from rent_ranked
    where rn = 1
), terms_active as (
    select distinct
        cast(t.n_agr as string) as n_agr,
        cast(t.c_nmrc as string) as c_nmrc,
        cast(t.d_valid_from as date) as d_valid_from,
        cast(t.d_valid_to as date) as d_valid_to
    from {agr_terms_table} t
    where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and t.c_nmrc is not null
      and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
      and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
), agreements_active as (
    select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to
    from {agreements_table} a
    where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
), companies_active as (
    select distinct
        cast(c.n_cmp as string) as n_cmp,
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
    from {companies_table} c
    where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
), mapped_raw as (
    select
        r.c_nmrc,
        r.d_rent_dt,
        r.n_amt_num,
        c.inn_key,
        cast(a.agr_id as string) as agr_id_key
    from rent_dedup r
    left join terms_active t
      on t.c_nmrc = r.c_nmrc
     and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
    left join agreements_active a
      on a.n_agr = t.n_agr
     and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
    left join companies_active c
      on c.n_cmp = a.n_cmp_client
)
"""

if not access_ok:
    print('SKIP: нет доступа к таблице.')
    mapping_coverage_df = pd.DataFrame()
    lake_april_key_df = pd.DataFrame(columns=['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_lake'])
else:
    sql_mapping_coverage = f"""
    {april_mapping_cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    )
    select
        count(*) as rent_rows_after_dedup,
        sum(n_amt_num) as total_n_amt,
        sum(case when valid_key_cnt = 0 then 1 else 0 end) as no_mapping_rows,
        sum(case when valid_key_cnt = 0 then n_amt_num else 0 end) as no_mapping_n_amt,
        sum(case when valid_key_cnt = 1 then 1 else 0 end) as unique_mapping_rows,
        sum(case when valid_key_cnt = 1 then n_amt_num else 0 end) as unique_mapping_n_amt,
        sum(case when valid_key_cnt > 1 then 1 else 0 end) as ambiguous_mapping_rows,
        sum(case when valid_key_cnt > 1 then n_amt_num else 0 end) as ambiguous_mapping_n_amt
    from map_stats
    """

    mapping_coverage_df = run_sql(sql_mapping_coverage, step_name='mapping_coverage_april')
    if len(mapping_coverage_df):
        total_rows = float(mapping_coverage_df.loc[0, 'rent_rows_after_dedup'] or 0)
        total_amt = float(mapping_coverage_df.loc[0, 'total_n_amt']) if pd.notna(mapping_coverage_df.loc[0, 'total_n_amt']) else np.nan
        if total_rows:
            mapping_coverage_df['no_mapping_rows_pct'] = mapping_coverage_df['no_mapping_rows'] / total_rows * 100.0
            mapping_coverage_df['unique_mapping_rows_pct'] = mapping_coverage_df['unique_mapping_rows'] / total_rows * 100.0
            mapping_coverage_df['ambiguous_mapping_rows_pct'] = mapping_coverage_df['ambiguous_mapping_rows'] / total_rows * 100.0
        if pd.notna(total_amt) and total_amt != 0:
            mapping_coverage_df['no_mapping_n_amt_pct'] = mapping_coverage_df['no_mapping_n_amt'] / total_amt * 100.0
            mapping_coverage_df['unique_mapping_n_amt_pct'] = mapping_coverage_df['unique_mapping_n_amt'] / total_amt * 100.0
            mapping_coverage_df['ambiguous_mapping_n_amt_pct'] = mapping_coverage_df['ambiguous_mapping_n_amt'] / total_amt * 100.0

    print('Mapping coverage:')
    display(mapping_coverage_df)

    sql_lake_key_april = f"""
    {april_mapping_cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    ), mapped_unique as (
        select
            mr.c_nmrc,
            mr.d_rent_dt,
            mr.n_amt_num,
            max(mr.inn_key) as inn_key,
            max(mr.agr_id_key) as agr_id_key
        from mapped_raw mr
        join map_stats ms
          on ms.c_nmrc = mr.c_nmrc
         and ms.d_rent_dt = mr.d_rent_dt
         and ms.n_amt_num = mr.n_amt_num
        where ms.valid_key_cnt = 1
          and mr.inn_key is not null
          and mr.agr_id_key is not null
        group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
    )
    select
        '{month_label}' as month_label,
        inn_key,
        agr_id_key,
        sum(n_amt_num) as commission_monthly_lake
    from mapped_unique
    group by inn_key, agr_id_key
    """

    lake_april_key_df = run_sql(sql_lake_key_april, step_name='lake_april_key_agg')
    if len(lake_april_key_df):
        lake_april_key_df['inn_key'] = lake_april_key_df['inn_key'].apply(normalize_inn_q1)
        lake_april_key_df['agr_id_key'] = lake_april_key_df['agr_id_key'].apply(normalize_agr_q1)
        lake_april_key_df['commission_monthly_lake'] = pd.to_numeric(
            lake_april_key_df['commission_monthly_lake'], errors='coerce'
        )
        lake_april_key_df = (
            lake_april_key_df.dropna(subset=['inn_key', 'agr_id_key'])
            .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
            .agg(commission_monthly_lake=('commission_monthly_lake', 'sum'))
        )

    print('Lake keys (unique mapping):', len(lake_april_key_df))
    display(lake_april_key_df.head(30))

## 3) Excel апрель: агрегат по `inn+agr_id`

In [ ]:
excel_col_map = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'comm_monthly_col': [
        'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)',
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия (руб в месяц)',
    ],
    'tariff_col': ['Тариф', 'Тарифный план', 'tariff_name'],
}


def first_nonempty_tariff(series):
    for v in series:
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s and s.lower() not in {'nan', 'none', 'null'}:
            return s
    return None


ex_raw = pd.read_excel(excel_april_path, header=excel_header_april)
resolved_excel_cols = {k: pick_col_robust(ex_raw.columns, v) for k, v in excel_col_map.items()}
missing_excel_cols = [k for k, v in resolved_excel_cols.items() if v is None]
if missing_excel_cols:
    raise ValueError(
        f'Не найдены колонки Excel: {missing_excel_cols}. Доступные: {list(ex_raw.columns)}'
    )

ex = ex_raw.copy()
ex['inn_key'] = ex[resolved_excel_cols['inn_col']].apply(normalize_inn_q1)
ex['agr_id_key'] = ex[resolved_excel_cols['agr_col']].apply(normalize_agr_q1)
ex['commission_monthly_excel'] = to_num_series(ex[resolved_excel_cols['comm_monthly_col']])
ex['tariff_name_excel'] = ex[resolved_excel_cols['tariff_col']]
ex['month_label'] = month_label

excel_april_key_df = (
    ex.dropna(subset=['inn_key', 'agr_id_key'])
      .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
      .agg(
          commission_monthly_excel=('commission_monthly_excel', 'max'),
          tariff_name_excel=('tariff_name_excel', first_nonempty_tariff),
      )
)

print('Excel resolved columns:', resolved_excel_cols)
print('Excel keys:', len(excel_april_key_df))
display(excel_april_key_df.head(30))


## 4) Сверка + match-rate + TOP mismatches

In [ ]:
compare_april_key_df = lake_april_key_df.merge(
    excel_april_key_df[['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel']],
    on=['month_label', 'inn_key', 'agr_id_key'],
    how='outer',
    indicator=True,
)
compare_april_key_df = compare_april_key_df.rename(columns={'_merge': 'merge_status'})

compare_april_key_df['commission_monthly_lake'] = compare_april_key_df['commission_monthly_lake'].fillna(0.0)
compare_april_key_df['commission_monthly_excel'] = compare_april_key_df['commission_monthly_excel'].fillna(0.0)
compare_april_key_df['delta_abs'] = (
    compare_april_key_df['commission_monthly_lake'] - compare_april_key_df['commission_monthly_excel']
)
compare_april_key_df['delta_pct'] = np.where(
    compare_april_key_df['commission_monthly_excel'] != 0,
    compare_april_key_df['delta_abs'].abs() / compare_april_key_df['commission_monthly_excel'].abs() * 100.0,
    np.nan,
)
compare_april_key_df['is_exact_match'] = compare_april_key_df['delta_abs'].abs() < exact_abs_tol
compare_april_key_df['is_near_match'] = (
    (compare_april_key_df['delta_abs'].abs() < near_abs_tol)
    | (compare_april_key_df['delta_pct'].fillna(np.inf) < near_pct_tol)
)

compare_april_key_df = compare_april_key_df.sort_values(
    by='delta_abs', key=lambda s: s.abs(), ascending=False
).reset_index(drop=True)

intersection_df = compare_april_key_df[compare_april_key_df['merge_status'] == 'both'].copy()
intersection_n = int(len(intersection_df))
exact_n = int(intersection_df['is_exact_match'].sum()) if intersection_n else 0
near_n = int(intersection_df['is_near_match'].sum()) if intersection_n else 0

excel_total = float(excel_april_key_df['commission_monthly_excel'].fillna(0).sum()) if len(excel_april_key_df) else 0.0
lake_total = float(lake_april_key_df['commission_monthly_lake'].fillna(0).sum()) if len(lake_april_key_df) else 0.0
inter_excel_total = float(intersection_df['commission_monthly_excel'].fillna(0).sum()) if intersection_n else 0.0
inter_lake_total = float(intersection_df['commission_monthly_lake'].fillna(0).sum()) if intersection_n else 0.0

summary_df = pd.DataFrame([
    {
        'month_label': month_label,
        'excel_key_cnt': int(len(excel_april_key_df)),
        'lake_key_cnt': int(len(lake_april_key_df)),
        'intersection_key_cnt': intersection_n,
        'only_excel_key_cnt': int((compare_april_key_df['merge_status'] == 'right_only').sum()),
        'only_lake_key_cnt': int((compare_april_key_df['merge_status'] == 'left_only').sum()),
        'intersection_share_of_excel_pct': (
            intersection_n / len(excel_april_key_df) * 100.0 if len(excel_april_key_df) else np.nan
        ),
        'intersection_share_of_lake_pct': (
            intersection_n / len(lake_april_key_df) * 100.0 if len(lake_april_key_df) else np.nan
        ),
        'exact_match_cnt': exact_n,
        'exact_match_rate_pct': exact_n / intersection_n * 100.0 if intersection_n else np.nan,
        'near_match_cnt': near_n,
        'near_match_rate_pct': near_n / intersection_n * 100.0 if intersection_n else np.nan,
        'excel_total': excel_total,
        'lake_total_unique_mapping': lake_total,
        'total_delta_abs': lake_total - excel_total,
        'total_delta_pct': abs(lake_total - excel_total) / abs(excel_total) * 100.0 if excel_total else np.nan,
        'intersection_excel_total': inter_excel_total,
        'intersection_lake_total': inter_lake_total,
        'intersection_delta_abs': inter_lake_total - inter_excel_total,
        'intersection_delta_pct': (
            abs(inter_lake_total - inter_excel_total) / abs(inter_excel_total) * 100.0
            if inter_excel_total else np.nan
        ),
        'exact_abs_tol': exact_abs_tol,
        'near_abs_tol': near_abs_tol,
        'near_pct_tol': near_pct_tol,
    }
])

merge_status_stats_df = (
    compare_april_key_df.groupby('merge_status', as_index=False)
    .agg(
        key_cnt=('inn_key', 'count'),
        excel_sum=('commission_monthly_excel', 'sum'),
        lake_sum=('commission_monthly_lake', 'sum'),
    )
)

top_mismatches_df = (
    intersection_df[~intersection_df['is_exact_match']]
    .sort_values(by='delta_abs', key=lambda s: s.abs(), ascending=False)
    .head(200)
    .copy()
)

print('=== SUMMARY ===')
display(summary_df)
print('Coverage by merge status:')
display(merge_status_stats_df)
print('TOP-200 mismatches on intersection (not exact):')
display(top_mismatches_df)

## 5) Right-only Excel (без Lake): разбивка по тарифам

Ключи только в Excel (`merge_status = right_only`) — клиенты без маппинга в `MPOS_RENT`.
Ожидаем нулевую комиссию; ниже — count и % в разрезе тарифа Excel.


In [ ]:
right_only_df = compare_april_key_df[compare_april_key_df['merge_status'] == 'right_only'].copy()
right_only_n = int(len(right_only_df))
excel_all_n = int(len(excel_april_key_df))

right_only_df['tariff_name_excel'] = right_only_df['tariff_name_excel'].fillna('(пусто)')
right_only_df.loc[
    right_only_df['tariff_name_excel'].astype(str).str.strip().isin({'', 'nan', 'None', 'null'}),
    'tariff_name_excel'
] = '(пусто)'

zero_cnt = int((right_only_df['commission_monthly_excel'].fillna(0).abs() < 0.01).sum()) if right_only_n else 0
nonzero_cnt = right_only_n - zero_cnt
zero_pct = zero_cnt / right_only_n * 100.0 if right_only_n else np.nan

right_only_zero_sanity_df = pd.DataFrame([{
    'right_only_key_cnt': right_only_n,
    'zero_commission_cnt': zero_cnt,
    'nonzero_commission_cnt': nonzero_cnt,
    'zero_commission_pct': zero_pct,
}])

tariff_right_only_df = (
    right_only_df
    .groupby('tariff_name_excel', dropna=False, as_index=False)
    .agg(client_cnt=('agr_id_key', 'count'))
    .sort_values('client_cnt', ascending=False)
    .reset_index(drop=True)
)

if right_only_n:
    tariff_right_only_df['pct_of_right_only'] = tariff_right_only_df['client_cnt'] / right_only_n * 100.0
else:
    tariff_right_only_df['pct_of_right_only'] = np.nan

if excel_all_n:
    tariff_right_only_df['pct_of_all_excel'] = tariff_right_only_df['client_cnt'] / excel_all_n * 100.0
else:
    tariff_right_only_df['pct_of_all_excel'] = np.nan

tariff_right_only_df['month_label'] = month_label
tariff_right_only_df = tariff_right_only_df[
    ['month_label', 'tariff_name_excel', 'client_cnt', 'pct_of_right_only', 'pct_of_all_excel']
]

print('Sanity: right_only commission == 0?')
display(right_only_zero_sanity_df)
print(f'Right-only by tariff (sum client_cnt={int(tariff_right_only_df["client_cnt"].sum()):,}):')
display(tariff_right_only_df)


## 6) Выгрузка отчёта


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_report_path, engine='xlsxwriter') as writer:
    summary_df.to_excel(writer, sheet_name='summary', index=False)
    merge_status_stats_df.to_excel(writer, sheet_name='coverage', index=False)
    if len(mapping_coverage_df):
        mapping_coverage_df.to_excel(writer, sheet_name='mapping_coverage', index=False)
    top_mismatches_df.to_excel(writer, sheet_name='top_mismatches', index=False)
    right_only_zero_sanity_df.to_excel(writer, sheet_name='right_only_sanity', index=False)
    tariff_right_only_df.to_excel(writer, sheet_name='tariff_right_only', index=False)
    compare_april_key_df.to_excel(writer, sheet_name='all_keys', index=False)

print(f'Report saved: {output_report_path}')
print(
    f"exact_match_rate={summary_df.loc[0, 'exact_match_rate_pct']:.2f}% | "
    f"near_match_rate={summary_df.loc[0, 'near_match_rate_pct']:.2f}% | "
    f"intersection_delta_pct={summary_df.loc[0, 'intersection_delta_pct']:.2f}%"
    if intersection_n and pd.notna(summary_df.loc[0, 'exact_match_rate_pct'])
    else 'No intersection keys to score.'
)
print(
    f"right_only={right_only_n:,}; zero_commission_pct={zero_pct:.2f}%; "
    f"tariff_groups={len(tariff_right_only_df):,}"
    if right_only_n
    else 'No right_only keys.'
)
